In [45]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.conexionDB import getEngine

In [46]:
motor = getEngine()

query = """
SELECT
    f.id_fact_contrato,
    f.numero_contrato,
    f.numero_proceso,
    f.valor_contratado,
    f.objeto_del_proceso,
    f.objeto_a_contratar,
    f.origen,

    tf.fecha        AS fecha_firma,
    ti.fecha        AS fecha_inicio,
    tfn.fecha       AS fecha_fin,

    e.codigo_entidad_en_secop,
    e.nombre_entidad,
    e.nit_entidad,
    e.nivel_entidad,
    e.departamento,
    e.municipio,

    p.nombre_proveedor,
    p.tipo_documento,
    p.documento,

    m.modalidad,
    s.estado

FROM analisishistorico.fact_contrato f

-- Fecha firma
LEFT JOIN analisishistorico.dim_tiempo tf 
    ON f.id_fecha_firma = tf.id_fecha

-- Fecha inicio
LEFT JOIN analisishistorico.dim_tiempo ti 
    ON f.id_fecha_inicio = ti.id_fecha

-- Fecha fin
LEFT JOIN analisishistorico.dim_tiempo tfn 
    ON f.id_fecha_fin = tfn.id_fecha

-- Dimensiones
LEFT JOIN analisishistorico.dim_entidad e 
    ON f.id_entidad = e.id_entidad

LEFT JOIN analisishistorico.dim_proveedor p 
    ON f.id_proveedor = p.id_proveedor

LEFT JOIN analisishistorico.dim_modalidad m 
    ON f.id_modalidad = m.id_modalidad

LEFT JOIN analisishistorico.dim_estado_proceso s 
    ON f.id_estado = s.id_estado;

"""
secop_historico = pd.read_sql(query, motor)


In [47]:
secop_historico.shape


(1665766, 21)

Total de registros

In [48]:
total_registros = secop_historico.shape[0]
total_registros



1665766

Contrato unicos

In [49]:
contratos_unicos = secop_historico['numero_contrato'].nunique()
contratos_unicos


1157865

Procesos unicos

In [50]:
procesos_unicos = secop_historico['numero_proceso'].nunique()
procesos_unicos


825201

Promedio de registros por contrato

In [51]:
secop_historico.groupby('numero_contrato').size().describe()


count    1.157865e+06
mean     1.438653e+00
std      8.820032e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      6.000000e+01
dtype: float64

Distribucion de estados

In [52]:
estado_dist = (
    secop_historico['estado']
    .value_counts(normalize=True)
    .mul(100)
    .round(3)
)

estado_dist


estado
EN EJECUCIÓN              27.462
MODIFICADO                25.831
ACTIVO                    19.762
CELEBRADO                 10.830
APROBADO                   8.207
TERMINADO                  4.139
CERRADO                    2.380
SUSPENDIDO                 0.648
CEDIDO                     0.599
LIQUIDADO                  0.098
TERMINADO SIN LIQUIDAR     0.037
CONVOCADO                  0.005
BORRADOR                   0.000
EN APROBACIÓN              0.000
ENVIADO PROVEEDOR          0.000
ADJUDICADO                 0.000
CANCELADO                  0.000
NO DEFINIDO                0.000
Name: proportion, dtype: float64

Estados por entidad

In [53]:
estado_por_entidad = (
    secop_historico
    .groupby(['nombre_entidad', 'estado'])
    .size()
    .reset_index(name='cantidad')
    .sort_values('cantidad', ascending=False)
)

estado_por_entidad.head(20)


,nombre_entidad,estado,cantidad
18621,SUBRED INTEGRADA DE SERVICIO DE SALUD CENTRO O...,MODIFICADO,30203
18642,SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR OCC...,MODIFICADO,28364
18627,SUBRED INTEGRADA DE SERVICIOS DE SALUD NORTE E...,MODIFICADO,22838
12629,INSTITUTO TECNOLOGICO METROPOLITANO,EN EJECUCIÓN,15031
18634,SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR E.S.E.,MODIFICADO,14809
17903,SECRETARÍA DISTRITAL DE INTEGRACIÓN SOCIAL,ACTIVO,7735
18641,SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR OCC...,EN EJECUCIÓN,6771
18633,SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR E.S.E.,EN EJECUCIÓN,6500
114,AGENCIA NACIONAL DE TIERRAS - ANT,ACTIVO,6237
17904,SECRETARÍA DISTRITAL DE INTEGRACIÓN SOCIAL,APROBADO,5293


Estados por contrato

In [54]:
estados_por_contrato = (
    secop_historico
    .groupby('numero_contrato')['estado']
    .nunique()
)

estados_por_contrato.describe()


count    1.157865e+06
mean     1.333868e+00
std      5.369881e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      5.000000e+00
Name: estado, dtype: float64

Fechas nulas por estado

In [56]:
fechas_inicio_nulas = (
    secop_historico
    .loc[secop_historico['fecha_inicio'].isna()]
    ['estado']
    .value_counts()
)

fechas_inicio_nulas


estado
ACTIVO               328642
APROBADO             136473
EN APROBACIÓN             5
BORRADOR                  4
ENVIADO PROVEEDOR         3
Name: count, dtype: int64

Fechas incoherentes

In [57]:
fechas_incoherentes = secop_historico[
    (secop_historico['fecha_inicio'].notna()) &
    (secop_historico['fecha_fin'].notna()) &
    (secop_historico['fecha_fin'] < secop_historico['fecha_inicio'])
]

fechas_incoherentes.shape


(0, 21)

Porcentaje por contratacion 

In [58]:
modalidad_dist = (
    secop_historico['modalidad']
    .value_counts(normalize=True)
    .mul(100)
    .round(3)
)

modalidad_dist


modalidad
CONTRATACIÓN DIRECTA                                                                     63.663
CONTRATACIÓN RÉGIMEN ESPECIAL                                                            16.993
CONTRATACIÓN DIRECTA (LEY 1150 DE 2007)                                                   7.548
MÍNIMA CUANTÍA                                                                            4.069
RÉGIMEN ESPECIAL                                                                          1.897
CONTRATACIÓN MÍNIMA CUANTÍA                                                               0.999
SELECCIÓN ABREVIADA DE MENOR CUANTÍA                                                      0.983
CONTRATACIÓN DIRECTA (CON OFERTAS)                                                        0.914
SELECCIÓN ABREVIADA SUBASTA INVERSA                                                       0.809
CONTRATACIÓN RÉGIMEN ESPECIAL (CON OFERTAS)                                               0.773
CONTRATOS Y CONVENIOS CON MÁS 

Entidades mas activas

In [59]:
entidades_activas = (
    secop_historico['nombre_entidad']
    .value_counts()
    .head(10)
)

entidades_activas


nombre_entidad
SUBRED INTEGRADA DE SERVICIO DE SALUD CENTRO ORIENTE E.S.E 1         37581
SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR OCCIDENTE ESE.            35891
SUBRED INTEGRADA DE SERVICIOS DE SALUD NORTE E.S.E. (OFICIAL         28927
INSTITUTO TECNOLOGICO METROPOLITANO                                  23282
SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR E.S.E.                    21973
SECRETARÍA DISTRITAL DE INTEGRACIÓN SOCIAL                           18443
AGENCIA NACIONAL DE TIERRAS - ANT                                    12783
ALCALDÍA DEL DISTRITO TURÍSTICO Y CULTURAL DE CARTAGENA DE INDIAS    12664
DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA             11082
GOBERNACIÓN DE BOYACÁ                                                10997
Name: count, dtype: int64

Por contratos unicos, entidades mas activas

In [60]:
entidades_contratos = (
    secop_historico
    .drop_duplicates('numero_contrato')
    ['nombre_entidad']
    .value_counts()
    .head(10)
)

entidades_contratos


nombre_entidad
INSTITUTO TECNOLOGICO METROPOLITANO                                  17314
SECRETARÍA DISTRITAL DE INTEGRACIÓN SOCIAL                           13849
AGENCIA NACIONAL DE TIERRAS - ANT                                    10434
DEFENSORÍA DEL PUEBLO                                                 8592
ALCALDÍA DEL DISTRITO TURÍSTICO Y CULTURAL DE CARTAGENA DE INDIAS     8548
DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA              8530
SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR OCCIDENTE ESE.             7491
SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR E.S.E.                     7322
SUBRED INTEGRADA DE SERVICIO DE SALUD CENTRO ORIENTE E.S.E 1          7191
SUBRED INTEGRADA DE SERVICIOS DE SALUD NORTE E.S.E. (OFICIAL          6909
Name: count, dtype: int64

In [62]:
resumen_historico = {
    'total_registros': total_registros,
    'contratos_unicos': contratos_unicos,
    'procesos_unicos': procesos_unicos,
    'promedio_registros_por_contrato': total_registros / contratos_unicos,
    'estados_distintos': secop_historico['estado'].nunique(),
    'modalidades_distintas': secop_historico['modalidad'].nunique()
}

pd.Series(resumen_historico)


total_registros                    1.665766e+06
contratos_unicos                   1.157865e+06
procesos_unicos                    8.252010e+05
promedio_registros_por_contrato    1.438653e+00
estados_distintos                  1.800000e+01
modalidades_distintas              2.600000e+01
dtype: float64

ANOMALIA: Contratos con demasiados cambios de estado

In [63]:
CLAVE_CONTRATO = [
    'numero_contrato',
    'documento',
    'codigo_entidad_en_secop'
]

cambios_estado = (
    secop_historico
    .groupby(CLAVE_CONTRATO)
    .size()
)

cambios_estado[cambios_estado > cambios_estado.quantile(0.99)]


numero_contrato     documento   codigo_entidad_en_secop
25-22-105118        890900286   205150011                  15
CO1.PCCNTR.7124184  901114890   700859218                   6
CO1.PCCNTR.7140570  7162508     702769076                   9
CO1.PCCNTR.7140666  24176583    702769076                   9
CO1.PCCNTR.7140956  19272009    702769076                   9
                                                           ..
CO1.PCCNTR.8461343  79825510    702769076                   6
CO1.PCCNTR.8461349  1021667983  702769076                   6
CO1.PCCNTR.8466548  1026599590  702769076                   6
CO1.PCCNTR.8474149  1013603180  702769076                   6
CO1.PCCNTR.8476665  1077840743  702769076                   6
Length: 9315, dtype: int64